# Network Analysis — Combiner · Other Projection · Predictor

Diagnostic notebook that probes what each learned component does and whether it is working.

| Section | Component | Key questions |
|---|---|---|
| 2 | **Combiner** | Is the gate s active? Does conditioning move embeddings toward the target? |
| 3 | **Other projection** | How far from identity? Does it distort or preserve the gallery space? |
| 4 | **Predictor** | Are predicted conditions spread and meaningful? Is it smooth and accurate? |
| 5 | **End-to-end** | Marginal contribution of each component on retrieval metrics. |


In [ ]:
EXPERIMENT_DIR = (
    # "/project/CoSiR/res/CoSiR_Experiment/impressions/20260519_195824_CoSiR_Experiment"
    # "/project/CoSiR/res/CoSiR_Experiment/impressions/20260520_023903_CoSiR_Experiment"
    "/project/CoSiR/res/CoSiR_Experiment/impressions/20260520_090015_CoSiR_Experiment"
)
TRAIN_JSON_PATH = "/project/Impressions/metadata/impressions_train.json"
EPOCH      = None   # None → latest
DEVICE     = "cuda"
TYPE_NAMES  = ["caption", "description", "impression", "aesthetic"]
TYPE_COLORS = ["#888888", "#2196F3", "#FF9800", "#4CAF50"]


In [ ]:
import os, sys, glob, json, warnings
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

warnings.filterwarnings("ignore")

import importlib.util as _ilu
def _load_file(name, path):
    spec = _ilu.spec_from_file_location(name, path)
    mod  = _ilu.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_proj = os.path.normpath(os.path.join(os.path.abspath(""), "../../.."))
_combiner_mod      = _load_file("combiner",             os.path.join(_proj, "src/model/combiner.py"))
Combiner_new       = _combiner_mod.Combiner_new
OtherProjMLP       = _combiner_mod.OtherProjMLP
ConditionPredictor = _load_file("condition_predictor",  os.path.join(_proj, "src/model/condition_predictor.py")).ConditionPredictor
print("Modules loaded")


In [ ]:
cond_dir = os.path.join(EXPERIMENT_DIR, "condition_viz")
ca_dir   = os.path.join(EXPERIMENT_DIR, "condition_analysis")

fixed = torch.load(os.path.join(cond_dir, "fixed_data.pt"), map_location="cpu", weights_only=False)
all_img_emb    = fixed["all_img_emb"]         # [N_img, 512]
all_txt_emb    = fixed["all_txt_emb"]         # [N_txt, 512]
all_raw_text   = fixed["all_raw_text"]
image_paths    = fixed["image_paths"]
img2txt        = fixed["image_to_text_map"]   # [N_img, cpi]
txt2img        = fixed["text_to_image_map"]   # [N_txt]
cpi            = fixed["captions_per_image"]
test_cap_types = fixed["test_caption_types"]  # [N_txt]
n_img, n_txt   = all_img_emb.shape[0], all_txt_emb.shape[0]
print(f"Test: {n_img} images × {cpi} caps = {n_txt} texts")

epoch_files = sorted(glob.glob(os.path.join(cond_dir, "epoch_*.pt")))
snap_path   = epoch_files[-1] if EPOCH is None else os.path.join(cond_dir, f"epoch_{EPOCH:04d}.pt")
snap        = torch.load(snap_path, map_location="cpu", weights_only=False)
epoch          = snap["epoch"]
label_emb_all  = snap["label_embeddings_all"]   # [N_train, D_cond]
representatives = snap["representatives"]         # [K, D_cond]
combine_side    = snap.get("combine_side", "img")
train_types     = snap.get("train_sample_types")  # [N_train]
K         = len(representatives)
label_dim = representatives.shape[1]
print(f"Epoch {epoch} | K={K} reps | label_dim={label_dim} | combine_side={combine_side}")

ca_files = sorted(glob.glob(os.path.join(ca_dir, "epoch_*.pt"))) if os.path.exists(ca_dir) else []
HAS_CA = False
if ca_files:
    ca_path = ca_files[-1] if EPOCH is None else os.path.join(ca_dir, f"epoch_{EPOCH:04d}.pt")
    ca = torch.load(ca_path, map_location="cpu", weights_only=False)
    per_rep_gt_rank  = ca["per_rep_gt_rank"]       # [K, N_txt]
    oracle_cond_idx  = ca["oracle_condition_idx"]  # [N_txt]
    clip_gt_rank     = ca["clip_gt_rank"]           # [N_txt]
    HAS_CA = True
    print(f"CA cache loaded (epoch {ca['epoch']})")

dev = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
cfg = snap["combiner_config"]
combiner = Combiner_new(
    clip_feature_dim=cfg["clip_feature_dim"],
    projection_dim=cfg["projection_dim"],
    label_dim=cfg["label_dim"],
    hidden_dim=cfg.get("hidden_dim", cfg["clip_feature_dim"]),
    num_heads=cfg.get("num_heads", 8),
    num_layers=cfg.get("num_layers", 2),
    dropout=cfg.get("dropout", 0.1),
).to(dev).eval()
combiner.load_state_dict(snap["combiner_state_dict"])

_fd      = cfg["clip_feature_dim"]
_op_cfg  = snap.get("other_proj_config", {})
_op_type = _op_cfg.get("type", "Linear")
if _op_type == "OtherProjMLP":
    other_proj = OtherProjMLP(
        feature_dim=_fd,
        hidden_dim=_op_cfg.get("hidden_dim", 1024),
        num_blocks=_op_cfg.get("num_blocks", 3),
    )
else:
    other_proj = torch.nn.Linear(_fd, _fd)
    torch.nn.init.eye_(other_proj.weight)
    torch.nn.init.zeros_(other_proj.bias)
if "other_proj_state_dict" in snap:
    other_proj.load_state_dict(snap["other_proj_state_dict"])
other_proj = other_proj.to(dev).eval()

with torch.no_grad():
    if combine_side == "txt":
        other_n = F.normalize(other_proj(all_img_emb.to(dev)), dim=-1).cpu()
    else:
        other_n = F.normalize(other_proj(all_txt_emb.to(dev)), dim=-1).cpu()

predictor = None
if "predictor_state_dict" in snap:
    pcfg = snap["predictor_config"]
    predictor = ConditionPredictor(
        input_dim=pcfg["input_dim"], hidden_dim=pcfg["hidden_dim"],
        output_dim=pcfg["output_dim"], num_layers=pcfg.get("num_layers", 2),
        dropout=pcfg.get("dropout", 0.1),
    ).to(dev).eval()
    predictor.load_state_dict(snap["predictor_state_dict"])
    print("Predictor loaded")

img_n = F.normalize(all_img_emb, dim=-1)
txt_n = F.normalize(all_txt_emb, dim=-1)
print("All models ready.")


In [ ]:
def r_at_k(ranks: torch.Tensor, k: int) -> float:
    return (ranks < k).float().mean().item() * 100

@torch.no_grad()
def apply_condition(emb: torch.Tensor, cond: torch.Tensor, bs: int = 256):
    out = []
    for i in range(0, emb.shape[0], bs):
        e = min(i + bs, emb.shape[0])
        out.append(combiner(emb[i:e].to(dev), None, cond.expand(e-i, -1).to(dev)))
    return F.normalize(torch.cat(out), dim=-1).cpu()

@torch.no_grad()
def apply_condition_return_scalar(emb: torch.Tensor, cond: torch.Tensor, bs: int = 256):
    """Returns (normalised_output [N,512], scalar [N,1])."""
    out_list, sc_list = [], []
    buf = []
    def _hook(m, inp, outp): buf.append(outp.detach().cpu())
    h = combiner.dynamic_scalar.register_forward_hook(_hook)
    with torch.no_grad():
        for i in range(0, emb.shape[0], bs):
            e = min(i + bs, emb.shape[0])
            buf.clear()
            o = combiner(emb[i:e].to(dev), None, cond.expand(e-i,-1).to(dev))
            out_list.append(o.cpu())
            sc_list.append(buf[0])
    h.remove()
    return F.normalize(torch.cat(out_list), dim=-1), torch.cat(sc_list)

@torch.no_grad()
def apply_condition_with_delta(emb: torch.Tensor, cond: torch.Tensor, bs: int = 256):
    """Returns (normalised_output, raw_delta, scalar) — all [N, D]."""
    out_list, delta_list, sc_list = [], [], []
    buf = []
    def _hook(m, inp, outp): buf.append(outp.detach().cpu())
    h = combiner.dynamic_scalar.register_forward_hook(_hook)
    with torch.no_grad():
        for i in range(0, emb.shape[0], bs):
            e = min(i + bs, emb.shape[0])
            buf.clear()
            o, d = combiner(emb[i:e].to(dev), None, cond.expand(e-i,-1).to(dev), return_delta=True)
            out_list.append(o.cpu()); delta_list.append(d.cpu()); sc_list.append(buf[0])
    h.remove()
    return (F.normalize(torch.cat(out_list), dim=-1),
            torch.cat(delta_list), torch.cat(sc_list))

def rank_after_condition(cond: torch.Tensor) -> torch.Tensor:
    if combine_side == "txt":
        combined = apply_condition(all_txt_emb, cond)
        sims = combined @ other_n.T
    else:
        combined = apply_condition(all_img_emb, cond)
        sims = other_n @ combined.T
    gt_s = sims[torch.arange(n_txt), txt2img]
    return (sims >= gt_s.unsqueeze(1)).sum(dim=1).long() - 1

# Pre-compute type means in condition space
if train_types is not None:
    type_means   = torch.stack([label_emb_all[train_types == t].mean(0) for t in range(4)])
    type_means_n = F.normalize(type_means, dim=-1)

# Pre-compute predicted conditions (combine_side determines input side)
if predictor is not None:
    src_emb = all_img_emb if combine_side == "img" else all_txt_emb
    with torch.no_grad():
        pred_conds = torch.cat([predictor(src_emb[i:i+512].to(dev)).cpu()
                                for i in range(0, src_emb.shape[0], 512)])
    print(f"Predicted conditions: {pred_conds.shape}")

print("Helpers ready.")


---
## 2 — Combiner Analysis

**Architecture recap**: `Combiner_new` compresses both the input embedding (512→128) and the condition (label_dim→128), concatenates them (256-d), and runs a gradual MLP to produce a `delta` (512-d). A learned scalar `s = sigmoid(...)` gates: `output = (1−s)·input + s·delta`. So `s≈0` means the input passes through unchanged; `s≈1` means the output is fully the delta.

### 2A — Dynamic Gate Scalar `s`

**Method**: A forward hook is registered on `combiner.dynamic_scalar` to capture the raw per-sample gate value during inference. The combiner is run on all combine-side embeddings once per representative, collecting `s` for every (input, condition) pair.

**Interpretation**:
- `s` is a per-sample sigmoid output that interpolates between the input and the learned delta: `output = (1−s)·input + s·delta`.
- The *overall distribution* shows whether the combiner is in passthrough mode or transformation mode globally.
- *Per-representative mean `s`*: some representatives may consistently trigger stronger gating than others — meaningful if different reps encode very different condition "strengths".
- *Per-sample std of `s` across reps*: high std means the gate is sensitive to which condition is applied (good); near-zero std means the gate is input-driven and condition-independent (condition barely matters for gating).

**Good**: `s` mean well above 0.3, with variation across representatives and inputs. Distribution is not a delta spike at 0 or 1.

**Bad**: `s` ≈ 0 everywhere (combiner never activates). Or `s` ≈ 1 everywhere with near-zero variance (gate saturated and identical for everything — lost input-sensitivity).


In [ ]:
# ── 2A: Dynamic scalar s distribution ────────────────────────────────────────
# Collect s per image for each representative.
# s≈0 → combiner ignores condition (passthrough); s≈1 → condition fully controls output.

reps_n = F.normalize(representatives, dim=-1)
combine_emb = all_img_emb if combine_side == "img" else all_txt_emb  # [N, 512]
N_combine = combine_emb.shape[0]

# Sample over all K reps; collecting scalar per sample per rep would be [K*N] — use mean per rep instead
scalars_per_rep = []  # [K] mean scalar
scalars_all = []      # collect from the mean-condition run
for ri in range(K):
    cond_r = representatives[ri:ri+1]
    _, sc = apply_condition_return_scalar(combine_emb, cond_r)
    scalars_per_rep.append(sc.squeeze(-1).numpy())

scalars_per_rep = np.stack(scalars_per_rep)  # [K, N_combine]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Overall distribution
ax = axes[0]
ax.hist(scalars_per_rep.flatten(), bins=60, color="#5c85d6", alpha=0.8)
ax.axvline(scalars_per_rep.mean(), color="red", lw=1.5, label=f"mean={scalars_per_rep.mean():.3f}")
ax.set_xlabel("Scalar s (gate value)")
ax.set_ylabel("Count")
ax.set_title("Distribution of gate scalar s\n(all images × all representatives)")
ax.legend()

# Per-rep mean scalar
ax2 = axes[1]
rep_means = scalars_per_rep.mean(axis=1)  # [K]
sort_idx = rep_means.argsort()
ax2.barh(range(K), rep_means[sort_idx], color="#5c85d6", alpha=0.8)
ax2.set_yticks(range(K)); ax2.set_yticklabels([f"Rep {sort_idx[i]}" for i in range(K)], fontsize=6)
ax2.set_xlabel("Mean scalar s"); ax2.set_title("Mean gate scalar per representative")
ax2.axvline(0.5, color="k", ls="--", lw=0.8, label="s=0.5")
ax2.legend(fontsize=7)

# Variance of s across reps per sample
ax3 = axes[2]
per_sample_std = scalars_per_rep.std(axis=0)  # [N_combine]
ax3.hist(per_sample_std, bins=40, color="#e07b54", alpha=0.8)
ax3.set_xlabel("Std of s across all representatives")
ax3.set_title("How much does s vary across reps\nfor the same input?")
ax3.axvline(per_sample_std.mean(), color="red", lw=1.5, label=f"mean={per_sample_std.mean():.3f}")
ax3.legend()

plt.tight_layout(); plt.show()

print(f"Global s: mean={scalars_per_rep.mean():.4f}  std={scalars_per_rep.std():.4f}  "
      f"min={scalars_per_rep.min():.4f}  max={scalars_per_rep.max():.4f}")
print(f"Interpretation: {'near-passthrough (condition barely used)' if scalars_per_rep.mean() < 0.3 else 'condition actively modulates output' if scalars_per_rep.mean() > 0.5 else 'moderate gating'}")


### 2B — Condition Sensitivity

**Method**: A fixed probe set of inputs is passed through the combiner with each of the K representatives in turn, producing K output embeddings per probe. Pairwise cosine similarity between all K outputs for the same input measures how differently each condition transforms the embedding.

**Interpretation**:
- *Pairwise cosine sim histogram*: the "spread" of the output space across conditions. Near 1.0 = conditions collapse to the same output; near 0.0 = maximally distinct outputs.
- *Cosine sim of output to input*: how much each representative shifts the embedding away from its origin. Sim ≈ 1.0 = almost no movement; sim ≈ 0.7 = substantial shift.
- *K×K heatmap*: clusters of reps that produce similar outputs may share a semantic direction.

**Good**: Mean pairwise sim clearly below 1.0 (e.g. 0.6–0.9). Shift-from-input varies across representatives. Heatmap shows structured groupings, not a uniform matrix.

**Bad**: All pairwise sims near 1.0 (the combiner ignores conditions — all reps produce the same output). Or all shifts near 1.0 (the combiner does nothing regardless of condition).


In [ ]:
# ── 2B: Condition sensitivity ─────────────────────────────────────────────────
# Fix a set of inputs; apply all K representatives; measure pairwise cosine sim
# of the outputs. Low similarity → conditions produce genuinely different outputs.

N_probe = min(200, N_combine)
probe_emb = combine_emb[:N_probe]

outputs_per_rep = []  # [K, N_probe, 512]
with torch.no_grad():
    for ri in range(K):
        cond_r = representatives[ri:ri+1]
        out = apply_condition(probe_emb, cond_r)
        outputs_per_rep.append(out)
outputs_per_rep = torch.stack(outputs_per_rep)  # [K, N_probe, 512]

# For each probe sample, compute pairwise cosine sim across K conditions
# Result: [N_probe, K, K]
out_n = F.normalize(outputs_per_rep, dim=-1)  # [K, N_probe, 512]
# For each probe, compute K×K cosine sim matrix across representatives
out_n_perm = out_n.permute(1, 0, 2)          # [N_probe, K, 512]
sim_across_reps = torch.bmm(out_n_perm, out_n_perm.transpose(1, 2))  # [N_probe, K, K]
upper_tri = sim_across_reps[:, torch.triu(torch.ones(K,K), diagonal=1).bool()]  # [N_probe, K*(K-1)/2]

# Also: compare to baseline (no condition = identity cond or zero cond)
# Baseline: normalized input (CLIP embedding unchanged)
baseline_n = F.normalize(probe_emb, dim=-1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.hist(upper_tri.flatten().numpy(), bins=50, color="#5c85d6", alpha=0.8)
ax.axvline(upper_tri.mean().item(), color="red", lw=1.5, label=f"mean={upper_tri.mean():.3f}")
ax.set_xlabel("Cosine sim between outputs of two different reps")
ax.set_title("Condition sensitivity:\ncosine sim of outputs across all rep pairs")
ax.legend()
print(f"Mean cosine sim between different-rep outputs: {upper_tri.mean():.4f}")
print(f"  (1.0=conditions produce identical outputs, 0.0=maximally different)")

# Per-rep: how much does each rep's output differ from the input?
ax2 = axes[1]
shift = []
for ri in range(K):
    s = (out_n[ri] * baseline_n).sum(dim=-1).numpy()
    shift.append(s.mean())
ax2.bar(range(K), shift, color="#5c85d6", alpha=0.8)
ax2.axhline(1.0, color="k", ls="--", lw=0.8, label="no change")
ax2.set_xlabel("Representative index")
ax2.set_ylabel("Mean cosine sim to input")
ax2.set_title("How much does each rep shift the embedding?\n(lower = bigger shift)")
ax2.legend(fontsize=7)
print("\nMean cosine sim of output to input (per rep):")
for ri, v in enumerate(shift): print(f"  Rep {ri:2d}: {v:.4f}")

# K×K heatmap: mean pairwise output similarity (averaged over all probe samples)
ax3 = axes[2]
mean_sim_kk = sim_across_reps.mean(0).numpy()  # [K, K]
im = ax3.imshow(mean_sim_kk, cmap="Blues", vmin=mean_sim_kk.min(), vmax=1)
ax3.set_title("Mean pairwise cosine sim of combiner\noutputs across representatives")
ax3.set_xlabel("Rep j"); ax3.set_ylabel("Rep i")
plt.colorbar(im, ax=ax3)

plt.tight_layout(); plt.show()


### 2C — Delta Alignment with Target Direction

**Method**: For each test input, the actual shift vector is `shift = combiner_output − input`. The ideal direction is `F.normalize(target − input)`, where `target` is the mean of the GT gallery embeddings (projected via `other_proj`). Cosine similarity between `shift` and `ideal` measures whether the combiner moves toward the right answer.

**Interpretation**:
- **+1.0**: combiner pushes the embedding *directly* toward its GT gallery match.
- **0.0**: shift is orthogonal to the ideal direction (neither helpful nor harmful).
- **−1.0**: combiner is moving *away* from the target.
- The fraction of samples with alignment > 0 shows consistency across the dataset.

**Good**: Mean alignment clearly positive (> 0.2), fraction > 0 close to 100%. Oracle condition should yield higher alignment than the average condition.

**Bad**: Mean near 0 or negative (combiner pushes embeddings in random or wrong directions). High variance with many negative values suggests the delta direction is unreliable.


In [ ]:
# ── 2C: Does the delta push toward the target? ────────────────────────────────
# For each test image, compute the combiner shift: (output - input).
# Then measure alignment with the ideal direction: (target_gallery - input).
# High cosine sim → combiner is moving embeddings in the right direction.
#
# combine_side="img": gallery=modified_images, query=other_proj(text)
#   Ideal shift direction for image i: other_n[GT_txt_j] - img_n[i]

if train_types is None:
    print("Skipped: needs train_types for type-mean condition")
else:
    # Use type-mean conditions (one per caption type)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax_idx, (cond_name, cond_vec) in enumerate([
        ("Type-mean (all types avg)", type_means.mean(0, keepdim=True)),
        ("Oracle per image (best rep)", None),
    ]):
        alignments = []
        scalars_c  = []

        if cond_vec is not None:
            # Single condition for all images
            out, delta, sc = apply_condition_with_delta(combine_emb, cond_vec)
            for img_idx in range(n_img):
                # GT texts for this image
                gt_txt_idxs = img2txt[img_idx]  # [cpi]
                target = other_n[gt_txt_idxs].mean(0)  # mean of GT text projections
                shift  = out[img_idx] - img_n[img_idx]  # direction of movement
                ideal  = F.normalize(target - img_n[img_idx], dim=0)
                shift_n = F.normalize(shift, dim=0)
                alignments.append((shift_n * ideal).sum().item())
                scalars_c.append(sc[img_idx].item())
        else:
            # Oracle: use per-image best representative from CA cache
            if not HAS_CA:
                print("No CA cache — skipping oracle condition alignment.")
                break
            # oracle_cond_idx is per-text; take the mode per image
            for img_idx in range(n_img):
                gt_txt_idxs = img2txt[img_idx]
                best_rep = oracle_cond_idx[gt_txt_idxs[0]].item()  # use first GT text
                cond_r = representatives[best_rep:best_rep+1]
                out_r, delta_r, sc_r = apply_condition_with_delta(
                    combine_emb[img_idx:img_idx+1], cond_r)
                target = other_n[gt_txt_idxs].mean(0)
                shift  = out_r[0] - img_n[img_idx]
                ideal  = F.normalize(target - img_n[img_idx], dim=0)
                shift_n = F.normalize(shift, dim=0)
                alignments.append((shift_n * ideal).sum().item())
                scalars_c.append(sc_r[0,0].item())

        ax = axes[ax_idx]
        ax.hist(alignments, bins=40, color="#4CAF50", alpha=0.8)
        ax.axvline(np.mean(alignments), color="red", lw=1.5,
                   label=f"mean={np.mean(alignments):.3f}")
        ax.axvline(0, color="k", ls="--", lw=0.8, label="perpendicular")
        ax.set_xlabel("Cosine sim: actual shift ↔ ideal direction (toward GT gallery)")
        ax.set_title(f"Delta alignment — {cond_name}\n>0 = combiner moves toward target")
        ax.legend(fontsize=7)
        print(f"{cond_name}: mean alignment={np.mean(alignments):.4f}  "
              f"fraction>0: {np.mean(np.array(alignments)>0)*100:.1f}%")

    plt.tight_layout(); plt.show()


### 2D — Output Geometry (PCA + Effective Rank)

**Method**: PCA is fit on the raw combine-side embeddings. The same projection is applied to the combiner's outputs under the average type-mean condition. **Effective rank** (participation ratio) is `PR = (Σσᵢ)² / Σσᵢ²` from the SVD of the centered embedding matrix — a continuous measure of how many dimensions are actually used.

**Interpretation**:
- PCA plots show whether conditioning reorganizes the embedding cloud geometry.
- Points are coloured by oracle-rep quartile (a proxy for which region of condition space matters for each input). Visible colour separation after conditioning (but not before) suggests the combiner amplifies semantically relevant structure.
- Effective rank: **higher = more spread/isotropic**; **lower = more collapsed** into fewer directions.

**Good**: Effective rank maintained or slightly increased. Any colour structure becomes more visible after conditioning. The point cloud does not shrink to a single cluster.

**Bad**: Effective rank drops dramatically (mode collapse — all outputs cluster together). PCA plots look identical before and after (combiner has no geometric effect).


In [ ]:
# ── 2D: Output geometry — PCA of embeddings before and after conditioning ─────
from sklearn.decomposition import PCA

if train_types is None:
    print("Skipped: needs train_types for type-mean conditions.")
else:
    # Compute combiner outputs under 3 conditions: passthrough (s→0 limit via zero cond),
    # type-mean averaged, oracle
    cond_avg = type_means.mean(0, keepdim=True)

    out_avg = apply_condition(combine_emb, cond_avg)

    # Stack: raw input + conditioned output
    raw_n = F.normalize(combine_emb, dim=-1).numpy()

    # PCA fit on raw input
    pca = PCA(n_components=2)
    pca.fit(raw_n)
    raw_pc   = pca.transform(raw_n)
    avg_pc   = pca.transform(out_avg.numpy())

    # If combine_side=img, we have one point per image; colour by majority caption type is N/A.
    # Instead colour images by their oracle representative index (bucketed into 4 groups).
    oracle_rep_per_img = oracle_cond_idx[img2txt[:, 0]].numpy() if HAS_CA else np.zeros(n_img, dtype=int)
    bucket = (oracle_rep_per_img / K * 4).astype(int).clip(0, 3)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (pc, title) in zip(axes, [(raw_pc, "Raw CLIP image embeddings"), (avg_pc, "After combiner (avg type-mean cond)")]):
        for b in range(4):
            m = bucket == b
            ax.scatter(pc[m, 0], pc[m, 1], s=10, alpha=0.5, color=TYPE_COLORS[b],
                       label=f"Oracle-rep quartile {b}")
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        ax.set_title(title)
        ax.legend(fontsize=7, markerscale=2)

    plt.tight_layout(); plt.show()

    # Effective rank (participation ratio): PR = (sum σ_i)^2 / sum σ_i^2
    def effective_rank(X):
        _, s, _ = np.linalg.svd(X - X.mean(0), full_matrices=False)
        s2 = s**2; s2 /= s2.sum()
        return float(np.exp(-np.sum(s2 * np.log(s2 + 1e-12))))

    raw_rank = effective_rank(raw_n)
    avg_rank = effective_rank(out_avg.numpy())
    print(f"Effective rank — Raw: {raw_rank:.1f}  |  After avg cond: {avg_rank:.1f}")
    print(f"  Lower effective rank = more collapsed/concentrated output space")


---
## 3 — Other Projection Analysis

`other_proj` transforms the non-combiner gallery/query side. It is either a `Linear(D→D)` (identity-initialized) or an `OtherProjMLP` (stack of residual blocks, identity-initialized). The question is: how far has it drifted from identity, and does that drift actually help?

Sections auto-detect which type is loaded.

### 3A — Weight / Structure Inspection

**Linear mode** — SVD is performed on the weight matrix `W ∈ ℝ^{D×D}` and the deviation `W − I`.
- **Singular values of W**: flat at 1 → pure rotation; values > 1 amplify, near 0 collapse.
- **Singular values of W − I**: effective rank of the learned perturbation. A few large values = focused low-rank change. Many large = dense full-rank noise.
- **Bias norm**: global shift of gallery space.

**MLP mode** — no single weight matrix, so three complementary views:
1. **Per-block Δ-weight norms** — Frobenius norm of each block's last linear layer (zero-initialized → norm=0 at start). Shows how much each residual block has learned.
2. **Effective Jacobian SVD** — `∂output/∂input` evaluated at the mean gallery embedding via autograd. Directly analogous to SVD-of-W; singular values near 1 = near-identity, far from 1 = learned warp.
3. **Embedding drift distribution** — L2 distance `‖other_proj(x) − x‖` over the gallery set. Shows how far the MLP has moved actual embeddings from their original positions.

**Good (both modes)**: moderate drift from identity, low-rank structure in the perturbation, no directions fully collapsed.

In [ ]:
# ── 3A: Weight / structure inspection ────────────────────────────────────────
is_mlp = isinstance(other_proj, OtherProjMLP)
_D = other_proj.feature_dim if is_mlp else other_proj.weight.shape[0]
print(f"other_proj type: {type(other_proj).__name__}  (D={_D})")

if not is_mlp:
    # ── Linear: classic SVD analysis ──────────────────────────────────────────
    W = other_proj.weight.data.cpu()   # [D, D]
    b = other_proj.bias.data.cpu()     # [D]

    frob_from_id = (W - torch.eye(_D)).norm(p="fro").item()
    bias_norm    = b.norm().item()
    print(f"Frobenius distance from identity: {frob_from_id:.4f}")
    print(f"Bias L2 norm:                     {bias_norm:.4f}")

    U, S, Vh     = torch.linalg.svd(W, full_matrices=False)
    _, S_delta, _ = torch.linalg.svd(W - torch.eye(_D), full_matrices=False)
    print(f"\nSingular values of W: min={S.min():.4f}  max={S.max():.4f}  "
          f"mean={S.mean():.4f}  cond={S.max()/S.min():.2f}")
    print(f"Singular values of (W−I): max={S_delta.max():.4f}  mean={S_delta.mean():.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(S.numpy(), color="#2196F3")
    axes[0].axhline(1.0, color="k", ls="--", lw=0.8, label="σ=1 (identity)")
    axes[0].set_xlabel("Index"); axes[0].set_ylabel("σ")
    axes[0].set_title("Singular values of W\n(flat=1 → pure rotation)"); axes[0].legend()
    axes[1].plot(S_delta.numpy(), color="#FF9800")
    axes[1].set_xlabel("Index"); axes[1].set_ylabel("σ")
    axes[1].set_title("Singular values of (W−I)\n(rank of learned perturbation)")
    axes[2].hist(b.numpy(), bins=40, color="#4CAF50", alpha=0.8)
    axes[2].axvline(0, color="k", ls="--", lw=0.8)
    axes[2].set_xlabel("Bias value"); axes[2].set_title("Bias distribution\n(0 = no systematic shift)")
    plt.suptitle("3A — Linear other_proj weight inspection", y=1.01)
    plt.tight_layout(); plt.show()

else:
    # ── MLP: per-block norms + Jacobian SVD + drift ────────────────────────────
    print("\n── Per-block Δ-weight norms (deviation from zero-init) ──")
    for bi, block in enumerate(other_proj.blocks):
        lw = block.dense_layers[-1].weight.data.cpu()
        lb = block.dense_layers[-1].bias.data.cpu()
        print(f"  Block {bi}: last-linear W_fro={lw.norm('fro'):.4f}  b_norm={lb.norm():.4f}")

    # Effective Jacobian at mean gallery embedding
    _gallery_raw = all_img_emb if combine_side == "txt" else all_txt_emb
    _x0 = torch.nn.functional.normalize(_gallery_raw, dim=-1).mean(0, keepdim=True).to(dev)
    _x0.requires_grad_(True)
    _J = torch.autograd.functional.jacobian(other_proj, _x0)  # [1,D,1,D]
    _J = _J.squeeze().cpu()                                     # [D, D]
    _x0.requires_grad_(False)

    _, S_jac, _   = torch.linalg.svd(_J, full_matrices=False)
    _, S_jdelta, _ = torch.linalg.svd(_J - torch.eye(_D), full_matrices=False)
    print(f"\nEffective Jacobian at mean embedding:")
    print(f"  Singular values: min={S_jac.min():.4f}  max={S_jac.max():.4f}  "
          f"mean={S_jac.mean():.4f}  cond={S_jac.max()/S_jac.min():.2f}")
    print(f"  (J−I) singular values: max={S_jdelta.max():.4f}  mean={S_jdelta.mean():.4f}")

    # Embedding drift on full gallery
    _bs = 512
    _drifts = []
    with torch.no_grad():
        for _i in range(0, _gallery_raw.shape[0], _bs):
            _xb = torch.nn.functional.normalize(_gallery_raw[_i:_i+_bs], dim=-1).to(dev)
            _yb = other_proj(_xb)
            _drifts.append((_yb - _xb).norm(dim=-1).cpu())
    _drifts = torch.cat(_drifts)
    print(f"\nEmbedding drift ‖other_proj(x)−x‖: "
          f"mean={_drifts.mean():.4f}  std={_drifts.std():.4f}  "
          f"max={_drifts.max():.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(S_jac.numpy(), color="#2196F3")
    axes[0].axhline(1.0, color="k", ls="--", lw=0.8, label="σ=1 (identity)")
    axes[0].set_xlabel("Index"); axes[0].set_ylabel("σ")
    axes[0].set_title("Effective Jacobian singular values\n(near 1 = near-identity)"); axes[0].legend()
    axes[1].plot(S_jdelta.numpy(), color="#FF9800")
    axes[1].set_xlabel("Index"); axes[1].set_ylabel("σ")
    axes[1].set_title("(J−I) singular values\n(rank of learned warp)")
    axes[2].hist(_drifts.numpy(), bins=50, color="#9C27B0", alpha=0.8)
    axes[2].axvline(0, color="k", ls="--", lw=0.8)
    axes[2].set_xlabel("‖other_proj(x) − x‖"); axes[2].set_title("Embedding drift distribution\n(0 = no change from identity)")
    plt.suptitle("3A — OtherProjMLP structure inspection", y=1.01)
    plt.tight_layout(); plt.show()


### 3B — Space Distortion (Pairwise Similarity Preservation)

**Method**: A random subsample of gallery embeddings is projected through `other_proj`. Cosine similarity is measured for all pairs *before* and *after* projection. Pearson correlation between these quantifies how well neighbourhood structure is preserved. Effective rank is compared before/after.

**Interpretation**:
- **Scatter plot**: points on the diagonal = no change. Above diagonal = pairs pulled closer; below = pushed apart.
- **Δ cosine sim distribution**: centered near 0 with low spread = structure-preserving with minor adjustments. Large positive mean = globally contracting (all pairs become more similar). Large negative = globally expanding.
- **Effective rank change**: increase = spreading into new dimensions; decrease = collapsing.

**Good**: Correlation > 0.8 (rough neighbourhood structure preserved) with some targeted deviations. Effective rank roughly maintained. Δ distribution symmetric around a small value.

**Bad**: Correlation near 0 (projection scrambles neighbourhood structure — gallery similarity is randomly reassigned). Large systematic shift in Δ distribution.


In [ ]:
# ── 3B: Space distortion — does other_proj preserve neighbour structure? ───────
# Compare pairwise cosine similarities before and after projection on a random subsample.

N_sub = min(400, (all_txt_emb if combine_side == "img" else all_img_emb).shape[0])
gallery_emb = all_txt_emb if combine_side == "img" else all_img_emb
idx_sub = torch.randperm(gallery_emb.shape[0])[:N_sub]
sub_emb = gallery_emb[idx_sub]

with torch.no_grad():
    sub_proj = F.normalize(other_proj(sub_emb.to(dev)), dim=-1).cpu()
sub_n = F.normalize(sub_emb, dim=-1)

# Pairwise cosine sims (upper triangle)
sim_before = (sub_n @ sub_n.T)
sim_after  = (sub_proj @ sub_proj.T)
mask_tri   = torch.triu(torch.ones(N_sub, N_sub), diagonal=1).bool()
sb = sim_before[mask_tri].numpy()
sa = sim_after[mask_tri].numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.scatter(sb[::5], sa[::5], s=2, alpha=0.3, color="#2196F3")
lo, hi = min(sb.min(), sa.min()), max(sb.max(), sa.max())
ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="no change")
ax.set_xlabel("Cosine sim before projection")
ax.set_ylabel("Cosine sim after projection")
ax.set_title("Pairwise similarity: before vs after\nother_proj (dots = pairs)")
ax.legend(fontsize=7)
corr = float(np.corrcoef(sb, sa)[0, 1])
print(f"Pairwise similarity correlation (before vs after): {corr:.4f}")
print(f"  (1.0=projection preserves all structure, <1.0=some reorganisation)")

ax2 = axes[1]
delta_sim = sa - sb
ax2.hist(delta_sim, bins=50, color="#FF9800", alpha=0.8)
ax2.axvline(0, color="k", ls="--", lw=0.8)
ax2.axvline(delta_sim.mean(), color="red", lw=1.5, label=f"mean={delta_sim.mean():.4f}")
ax2.set_xlabel("Δ cosine sim (after − before)")
ax2.set_title("Change in pairwise similarity\n(+= pairs pulled closer, −= pushed apart)")
ax2.legend(fontsize=7)

# Effective rank before vs after
ax3 = axes[2]
sub_n_np, sub_proj_np = sub_n.numpy(), sub_proj.numpy()
def eff_rank(X):
    _, s, _ = np.linalg.svd(X - X.mean(0), full_matrices=False)
    s2 = s**2 / (s**2).sum()
    return float(np.exp(-np.sum(s2 * np.log(s2 + 1e-12))))
er_before = eff_rank(sub_n_np)
er_after  = eff_rank(sub_proj_np)
ax3.bar(["Before proj", "After proj"], [er_before, er_after], color=["#888888","#2196F3"], alpha=0.8)
ax3.set_ylabel("Effective rank (participation ratio)")
ax3.set_title(f"Effective rank of gallery side\nbefore vs after other_proj")
for xi, v in enumerate([er_before, er_after]):
    ax3.text(xi, v + 0.5, f"{v:.1f}", ha="center")

plt.tight_layout(); plt.show()
print(f"Effective rank: before={er_before:.1f}  after={er_after:.1f}")


### 3C — Retrieval Ablation (Learned vs Identity Projection)

**Method**: The full pipeline is evaluated twice — once with the trained `other_proj`, once with an identity matrix substituted in. Both use the same combiner with the average type-mean condition. T2I R@1/5/10 is reported for each, plus the raw CLIP baseline.

**Interpretation**:
- Directly measures the **marginal contribution of `other_proj`** to retrieval, isolated from the combiner.
- "Combiner + identity" reveals whether the combiner alone (without projection adjustment) can beat CLIP.
- A positive gap (learned > identity) confirms the projection is learning useful alignment between the gallery side and the combiner's output space.

**Good**: Learned projection clearly outperforms identity by several R@1 points. Combiner + identity already beats CLIP (combiner is useful even without the projection). Learned projection adds a further lift.

**Bad**: Learned projection matches or underperforms identity (the 512×512 layer is wasted capacity). Combiner + identity is worse than CLIP (the combiner is hurting retrieval without the projection to compensate).


In [ ]:
# ── 3C: Retrieval ablation — learned projection vs identity ───────────────────
# Isolate the contribution of other_proj by swapping in an identity projection.

if train_types is None:
    print("Skipped: needs train_types for type-mean condition baseline")
else:
    cond_avg = type_means.mean(0, keepdim=True)

    # Method A: full pipeline (learned other_proj + combiner with avg cond)
    comb_out = apply_condition(combine_emb, cond_avg)
    if combine_side == "img":
        sims_A = other_n @ comb_out.T       # [N_txt, N_img]
    else:
        sims_A = comb_out @ other_n.T

    def get_gt_rank(sims):
        gt_s = sims[torch.arange(n_txt), txt2img]
        return (sims >= gt_s.unsqueeze(1)).sum(1).long() - 1

    rank_A = get_gt_rank(sims_A)

    # Method B: identity other_proj (replace other_n with normalized raw gallery)
    raw_other = F.normalize(all_txt_emb if combine_side == "img" else all_img_emb, dim=-1)
    if combine_side == "img":
        sims_B = raw_other @ comb_out.T
    else:
        sims_B = comb_out @ raw_other.T
    rank_B = get_gt_rank(sims_B)

    # Method C: CLIP baseline (no combiner, no other_proj)
    clip_sims = txt_n @ img_n.T
    gt_s_clip = clip_sims[torch.arange(n_txt), txt2img]
    rank_clip  = (clip_sims >= gt_s_clip.unsqueeze(1)).sum(1).long() - 1

    methods = [
        ("CLIP baseline",          rank_clip, "#bdbdbd"),
        ("Combiner + identity proj", rank_B, "#FF9800"),
        ("Combiner + learned proj",  rank_A, "#2196F3"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, k in zip(axes, [1, 5, 10]):
        names = [m[0] for m in methods]
        vals  = [r_at_k(m[1], k) for m in methods]
        cols  = [m[2] for m in methods]
        bars  = ax.bar(names, vals, color=cols, alpha=0.85)
        ax.bar_label(bars, fmt="%.1f%%", padding=2, fontsize=8)
        ax.set_title(f"R@{k}"); ax.set_ylabel("Recall (%)")
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=15, ha="right", fontsize=8)
    plt.suptitle("Ablation: contribution of other_proj (avg type-mean condition)")
    plt.tight_layout(); plt.show()

    print(f"\n{'Method':<30} {'R@1':>6} {'R@5':>6} {'R@10':>7}")
    print("-"*52)
    for name, ranks, _ in methods:
        print(f"{name:<30} {r_at_k(ranks,1):>6.1f} {r_at_k(ranks,5):>6.1f} {r_at_k(ranks,10):>7.1f}")
    delta_r1 = r_at_k(rank_A, 1) - r_at_k(rank_B, 1)
    print(f"\nother_proj marginal R@1 gain: {delta_r1:+.1f}pp")


---
## 4 — Predictor Analysis

The predictor is an MLP (`input_dim=512 → hidden → label_dim`) trained by stop-gradient distillation to mimic per-sample learned conditions. At test time it predicts a condition from the combine-side embedding alone — no access to caption text when `combine_side="img"`.

### 4A — Placement in Condition Space (PCA)

**Method**: PCA is fit jointly on a subsample of training conditions, all K representatives, and all predicted test conditions. All three sets are projected into this shared 2D space for visual comparison.

**Interpretation**:
- **Left plot** (training + reps): reveals the structure of the learned condition manifold. Type-clustered colours suggest the space has semantic organisation. Representatives (★) should be spread across the manifold to provide good coverage.
- **Right plot** (predicted + training backdrop): predicted conditions (pink) should overlap with the training manifold. If they land outside it, the predictor is extrapolating into regions where the combiner was never trained.

**Good**: Training conditions form soft type-clusters. Representatives are spread across different type regions. Predicted conditions overlap the training manifold and spread across it — not all collapsed to one corner.

**Bad**: Training conditions form a single undifferentiated blob (no semantic structure learned). Predicted conditions cluster far from most representatives (predictor always outputs the same region regardless of input — mode collapse).


In [ ]:
# ── 4A: Placement in condition space — PCA of training conds, reps, predictions ─
if predictor is None:
    print("Skipped: no predictor")
else:
    from sklearn.decomposition import PCA

    # Subsample training conditions for readability
    N_train_sub = min(2000, label_emb_all.shape[0])
    idx_tr = torch.randperm(label_emb_all.shape[0])[:N_train_sub]
    train_sub    = label_emb_all[idx_tr].numpy()
    train_types_sub = train_types[idx_tr].numpy() if train_types is not None else None

    pred_np = pred_conds.numpy()  # [N_combine, label_dim]
    reps_np = representatives.numpy()  # [K, label_dim]

    # Fit PCA on all training conditions
    all_for_pca = np.concatenate([train_sub, reps_np, pred_np])
    pca2 = PCA(n_components=2)
    pca2.fit(all_for_pca)

    tr_pc   = pca2.transform(train_sub)
    rep_pc  = pca2.transform(reps_np)
    pred_pc = pca2.transform(pred_np)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: training conditions coloured by type + representatives
    ax = axes[0]
    if train_types_sub is not None:
        for t in range(4):
            m = train_types_sub == t
            ax.scatter(tr_pc[m, 0], tr_pc[m, 1], s=6, alpha=0.3, color=TYPE_COLORS[t], label=TYPE_NAMES[t])
    else:
        ax.scatter(tr_pc[:, 0], tr_pc[:, 1], s=6, alpha=0.3, color="#888888", label="train conds")
    ax.scatter(rep_pc[:, 0], rep_pc[:, 1], s=80, marker="*", color="black", zorder=5, label="Representatives")
    for ri in range(K):
        ax.annotate(str(ri), rep_pc[ri], fontsize=6, ha="center")
    ax.set_title("Training conditions + Representatives")
    ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")
    ax.legend(fontsize=7, markerscale=2)

    # Right: predicted conditions (test images)
    ax2 = axes[1]
    if train_types_sub is not None:
        for t in range(4):
            m = train_types_sub == t
            ax2.scatter(tr_pc[m, 0], tr_pc[m, 1], s=4, alpha=0.15, color=TYPE_COLORS[t])
    ax2.scatter(rep_pc[:, 0], rep_pc[:, 1], s=80, marker="*", color="black", zorder=5, label="Reps")
    ax2.scatter(pred_pc[:, 0], pred_pc[:, 1], s=12, alpha=0.6, color="#e91e63", label="Predicted (test images)")
    ax2.set_title("Predicted conditions (test images) vs training space")
    ax2.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
    ax2.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")
    ax2.legend(fontsize=7, markerscale=2)

    plt.tight_layout(); plt.show()


### 4B — Prediction Diversity

**Method**: Pairwise cosine similarities are computed for three sets independently: predicted test conditions, a subsample of training conditions, and the K representatives. The distributions are compared — lower mean = more diverse set.

**Interpretation**:
- Pairwise sim near **1.0** for predicted conditions = **mode collapse**: the predictor outputs nearly the same vector for every input.
- Pairwise sim matching the training conditions = the predictor is covering the condition space similarly to how training samples did.
- Representatives typically have lower pairwise sim (they are intentionally spread as centroids).

**Good**: Predicted conditions have mean pairwise sim similar to or lower than training conditions. Distribution is spread, not a spike near 1.0.

**Bad**: Predicted conditions have much higher mean pairwise sim than training conditions — collapsed to a single mode. Or very wide distribution with many negative values (predictor outputs are random/incoherent).


In [ ]:
# ── 4B: Prediction diversity — are predicted conditions spread or collapsed? ───
if predictor is None:
    print("Skipped")
else:
    pred_np = pred_conds.numpy()
    train_np_full = label_emb_all.numpy()

    N_s = min(500, pred_np.shape[0])
    idx_p = np.random.choice(pred_np.shape[0], N_s, replace=False)
    idx_t = np.random.choice(train_np_full.shape[0], N_s, replace=False)
    sub_pred  = torch.tensor(pred_np[idx_p])
    sub_train = torch.tensor(train_np_full[idx_t])

    def pairwise_cosine(X):
        Xn = F.normalize(X, dim=-1)
        sim = (Xn @ Xn.T)
        mask = torch.triu(torch.ones(len(X), len(X)), diagonal=1).bool()
        return sim[mask].numpy()

    pw_pred  = pairwise_cosine(sub_pred)
    pw_train = pairwise_cosine(sub_train)
    pw_reps  = pairwise_cosine(representatives)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, (vals, label, col) in zip(axes, [
        (pw_pred,  "Predicted conds (test)",   "#e91e63"),
        (pw_train, "Training conds (sample)",   "#2196F3"),
        (pw_reps,  "Representatives",           "#4CAF50"),
    ]):
        ax.hist(vals, bins=40, color=col, alpha=0.8)
        ax.axvline(vals.mean(), color="red", lw=1.5, label=f"mean={vals.mean():.3f}")
        ax.set_xlabel("Pairwise cosine sim"); ax.set_ylabel("Count")
        ax.set_title(f"Diversity: {label}")
        ax.legend(fontsize=8)

    plt.tight_layout(); plt.show()

    print(f"Pairwise cosine sim (lower = more diverse):")
    print(f"  Predicted (test):   mean={pw_pred.mean():.4f}  std={pw_pred.std():.4f}")
    print(f"  Training conds:     mean={pw_train.mean():.4f}  std={pw_train.std():.4f}")
    print(f"  Representatives:    mean={pw_reps.mean():.4f}  std={pw_reps.std():.4f}")


### 4C — Predictor Smoothness (Neighbour Consistency)

**Method**: For each test input, its top-K nearest neighbours in CLIP embedding space are found (by cosine similarity). The cosine similarity between the predicted conditions of the input and each neighbour is measured, then compared against random pairs. The gap between the two distributions quantifies local smoothness.

**Interpretation**:
- A **large gap** (NN distribution shifted right relative to random) means similar CLIP inputs → similar predicted conditions — the predictor is a smooth, well-generalised function.
- A **zero gap** means the predictor is as erratic as random — CLIP-similar images get completely different predicted conditions (poor generalisation or memorisation without structure).

**Good**: NN distribution clearly right-shifted (gap > 0.1). The predictor generalises smoothly to unseen images.

**Bad**: The two distributions overlap almost entirely (gap ≈ 0). The predictor's output has no relationship to the input's neighbourhood structure — erratic, unreliable predictions.


In [ ]:
# ── 4C: Smoothness — do similar inputs get similar predictions? ───────────────
# For each test image, find its k nearest neighbours in CLIP space.
# Then measure: are predicted conditions of k-NN more similar than random pairs?

if predictor is None:
    print("Skipped")
else:
    src_np = F.normalize(src_emb, dim=-1).numpy()  # [N_combine, 512]
    pred_n = F.normalize(pred_conds, dim=-1)       # [N_combine, label_dim]

    K_nn = 10
    sim_input = torch.tensor(src_np) @ torch.tensor(src_np).T  # [N, N]
    # For each sample, get top-K neighbours (excluding self)
    nn_sim_pred, random_sim_pred = [], []
    N_c = src_np.shape[0]
    for i in range(N_c):
        sims_i = sim_input[i].clone(); sims_i[i] = -1
        nn_idx = sims_i.topk(K_nn).indices
        # Cosine sim of predicted conditions for neighbours
        p_i = pred_n[i]
        p_nn = pred_n[nn_idx]
        nn_sim_pred.extend((p_nn * p_i).sum(-1).tolist())

    # Random pairs
    rand_idx = torch.randint(0, N_c, (N_c * K_nn,))
    anchor_idx = torch.arange(N_c).repeat_interleave(K_nn)
    random_sim_pred = (pred_n[anchor_idx] * pred_n[rand_idx]).sum(-1).tolist()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(nn_sim_pred,    bins=50, alpha=0.65, color="#2196F3", label=f"Top-{K_nn} CLIP neighbours", density=True)
    ax.hist(random_sim_pred, bins=50, alpha=0.65, color="#888888", label="Random pairs",  density=True)
    ax.set_xlabel("Cosine sim between predicted conditions")
    ax.set_ylabel("Density")
    ax.set_title(f"Predictor smoothness: are CLIP neighbours' predicted conditions more similar?\n"
                 f"(gap between curves = predictor is smooth in input space)")
    ax.legend()
    plt.tight_layout(); plt.show()

    print(f"Predicted condition cosine sim:")
    print(f"  Top-{K_nn} CLIP neighbours: {np.mean(nn_sim_pred):.4f}")
    print(f"  Random pairs:               {np.mean(random_sim_pred):.4f}")
    gap = np.mean(nn_sim_pred) - np.mean(random_sim_pred)
    print(f"  Gap:                        {gap:+.4f}  {'(smooth)' if gap > 0.05 else '(little local structure)'}")


### 4D — Failure Mode Analysis

**Method**: Each test image is classified as *helps* / *neutral* / *hurts* based on whether the predictor's condition improves or degrades its mean T2I rank across all 4 captions vs the CLIP baseline. Image embeddings are projected to 2D PCA and coloured by outcome.

**Interpretation**:
- **Spatial clustering of failures (red)**: if hurt images concentrate in a specific PCA region, the predictor has a systematic blind spot for that image type. If scattered randomly, failures are noise-like and likely irreducible.
- **Rank improvement magnitude**: small symmetric hurt is less concerning than large catastrophic failures on specific images.

**Good**: Helped images substantially outnumber hurt ones. Red points scattered randomly (no systematic failure region). Hurt distribution narrow (small magnitude).

**Bad**: Hurt images cluster in a specific PCA region (systematic blind spot for a recognisable image type). Long tail in the hurt distribution (catastrophic rank regression on specific inputs). Hurt images outnumber helped ones.


In [ ]:
# ── 4D: Predictor failure modes ───────────────────────────────────────────────
# "Failure" = predictor condition leads to worse or equal T2I retrieval vs CLIP.
# Inspect: where do failed images sit in CLIP PCA space? What do they look like?

if predictor is None or not HAS_CA:
    print("Skipped: needs predictor + CA cache")
else:
    # Predictor rank (combine_side=img: pred_conds is [N_img, D_cond])
    with torch.no_grad():
        if combine_side == "img":
            pred_out = torch.cat([
                combiner(combine_emb[i:i+256].to(dev), None, pred_conds[i:i+256].to(dev)).cpu()
                for i in range(0, n_img, 256)])
            pred_out_n = F.normalize(pred_out, dim=-1)
            pred_sims  = other_n @ pred_out_n.T      # [N_txt, N_img]
        else:
            pred_out = torch.cat([
                combiner(combine_emb[i:i+256].to(dev), None, pred_conds[i:i+256].to(dev)).cpu()
                for i in range(0, n_txt, 256)])
            pred_out_n = F.normalize(pred_out, dim=-1)
            pred_sims  = pred_out_n @ other_n.T      # [N_txt, N_img]

    pred_gt_s    = pred_sims[torch.arange(n_txt), txt2img]
    pred_gt_rank = (pred_sims >= pred_gt_s.unsqueeze(1)).sum(1).long() - 1  # [N_txt]

    # Per-image: did predictor help? (compare to CLIP baseline)
    # Aggregate per-image: mean rank across its 4 texts
    pred_rank_per_img = torch.stack([pred_gt_rank[img2txt[:, c]] for c in range(cpi)]).float().mean(0)
    clip_rank_per_img = torch.stack([clip_gt_rank[img2txt[:, c]] for c in range(cpi)]).float().mean(0)

    helps  = (pred_rank_per_img < clip_rank_per_img).numpy()   # predictor improved rank
    hurts  = (pred_rank_per_img > clip_rank_per_img).numpy()   # predictor hurt rank
    neutral= ~helps & ~hurts

    print(f"Predictor vs CLIP (per image, mean rank across {cpi} captions):")
    print(f"  Helps:   {helps.sum()} ({helps.mean()*100:.1f}%)")
    print(f"  Neutral: {neutral.sum()} ({neutral.mean()*100:.1f}%)")
    print(f"  Hurts:   {hurts.sum()} ({hurts.mean()*100:.1f}%)")

    # PCA of image embeddings coloured by outcome
    pca_f = PCA(n_components=2)
    img_pc = pca_f.fit_transform(img_n.numpy())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.scatter(img_pc[neutral, 0], img_pc[neutral, 1], s=8, alpha=0.3, color="#888888", label=f"Neutral ({neutral.sum()})")
    ax.scatter(img_pc[helps, 0],   img_pc[helps, 1],   s=10, alpha=0.5, color="#4CAF50", label=f"Helps ({helps.sum()})")
    ax.scatter(img_pc[hurts, 0],   img_pc[hurts, 1],   s=10, alpha=0.5, color="#ef5350", label=f"Hurts ({hurts.sum()})")
    ax.set_title("Image CLIP embeddings (PCA)\ncoloured by predictor outcome")
    ax.set_xlabel(f"PC1 ({pca_f.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca_f.explained_variance_ratio_[1]*100:.1f}%)")
    ax.legend(fontsize=7, markerscale=2)

    # Rank improvement distribution
    ax2 = axes[1]
    improvement = (clip_rank_per_img - pred_rank_per_img).numpy()
    ax2.hist(improvement[helps],   bins=30, alpha=0.7, color="#4CAF50", label="Helps", density=False)
    ax2.hist(improvement[hurts],   bins=30, alpha=0.7, color="#ef5350", label="Hurts", density=False)
    ax2.axvline(0, color="k", ls="--", lw=0.8)
    ax2.set_xlabel("Rank improvement: CLIP − predictor rank (per image, mean over caps)")
    ax2.set_title("How much does predictor help/hurt?\n(positive = better than CLIP)")
    ax2.legend()

    plt.tight_layout(); plt.show()


---
## 5 — End-to-End Pipeline Analysis

### 5A — Contribution ladder
Marginal R@1/5/10 added by each component in order:
`CLIP → +other_proj → +combiner(zero cond) → +combiner(avg type-mean) → +combiner(predicted) → +combiner(oracle)`

### 5A — Contribution Ladder

**Method**: The pipeline is evaluated with each component added sequentially, holding the rest fixed:
1. **CLIP (raw)** — no learned components
2. **+ other_proj** — learned gallery-side projection only, no combiner
3. **+ combiner (zero cond)** — combiner with a zero condition vector (no semantic signal)
4. **+ combiner (avg type-mean)** — weak but real condition signal
5. **+ combiner (predicted)** — predictor's per-image condition at test time
6. **+ combiner (oracle)** — best possible representative per query (upper bound)

**Interpretation**:
- The gap between consecutive steps is each component's **marginal R@1 contribution**.
- The gap between *predicted* and *oracle* shows **headroom** for predictor improvement.
- A step that *decreases* recall means that component is harmful in isolation with the current setup.
- "Zero cond" vs "avg type-mean" reveals whether any semantic condition signal is needed, or whether the combiner's architecture alone changes retrieval.

**Good**: Monotonically increasing ladder. Oracle clearly above predicted (meaningful headroom). Predicted clearly above avg type-mean (predictor adds per-instance value). Zero-cond ≈ CLIP (combiner only activates with real conditions).

**Bad**: Non-monotonic ladder (a component hurts). Predicted ≈ oracle (either predictor is perfect or oracle is weak). Zero-cond much better than CLIP (combiner has a strong architectural bias independent of the condition — may be fitting noise).


In [ ]:
# ── 5A: Contribution ladder ───────────────────────────────────────────────────

results = {}

# Step 0: CLIP baseline
clip_sims_all = txt_n @ img_n.T
gt_s = clip_sims_all[torch.arange(n_txt), txt2img]
results["CLIP (raw)"] = ((clip_sims_all >= gt_s.unsqueeze(1)).sum(1).long() - 1, "#bdbdbd")

# Step 1: other_proj on gallery side only, no combiner
#   combine_side=img: gallery is images (unchanged), query is other_proj(txt)
#   — actually other_proj is on the NON-combiner side, which is txt when combine_side=img
#   sims = other_n @ img_n.T (other_n already computed with learned or identity proj)
if combine_side == "img":
    sims_proj = other_n @ img_n.T
else:
    sims_proj = txt_n @ other_n.T
gt_s_p = sims_proj[torch.arange(n_txt), txt2img]
results["+ other_proj (no combiner)"] = ((sims_proj >= gt_s_p.unsqueeze(1)).sum(1).long() - 1, "#9e9e9e")

# Step 2: combiner with zero condition (test if combiner alone helps without a real condition)
with torch.no_grad():
    zero_cond = torch.zeros(1, label_dim)
    out_zero = apply_condition(combine_emb, zero_cond)
    if combine_side == "img":
        sims_zero = other_n @ out_zero.T
    else:
        sims_zero = out_zero @ other_n.T
gt_s_z = sims_zero[torch.arange(n_txt), txt2img]
results["+ combiner (zero cond)"] = ((sims_zero >= gt_s_z.unsqueeze(1)).sum(1).long() - 1, "#FF9800")

# Step 3: combiner with average type-mean condition
if train_types is not None:
    cond_avg = type_means.mean(0, keepdim=True)
    out_avg = apply_condition(combine_emb, cond_avg)
    if combine_side == "img":
        sims_avg = other_n @ out_avg.T
    else:
        sims_avg = out_avg @ other_n.T
    gt_s_a = sims_avg[torch.arange(n_txt), txt2img]
    results["+ combiner (avg type-mean)"] = ((sims_avg >= gt_s_a.unsqueeze(1)).sum(1).long() - 1, "#FF9800")

# Step 4: combiner with predicted condition
if predictor is not None:
    results["+ combiner (predicted)"] = (pred_gt_rank, "#2196F3")

# Step 5: oracle (best representative per text)
if HAS_CA:
    results["+ combiner (oracle)"] = (per_rep_gt_rank.min(0).values, "#4CAF50")

# Table
print(f"{'Method':<36} {'R@1':>6} {'R@5':>6} {'R@10':>7}")
print("-" * 58)
for name, (ranks, _) in results.items():
    print(f"{name:<36} {r_at_k(ranks,1):>6.1f} {r_at_k(ranks,5):>6.1f} {r_at_k(ranks,10):>7.1f}")

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
names = list(results.keys())
colors = [v[1] for v in results.values()]
for ax, k in zip(axes, [1, 5, 10]):
    vals = [r_at_k(v[0], k) for v in results.values()]
    bars = ax.barh(names, vals, color=colors, alpha=0.85)
    ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=8)
    ax.set_xlabel(f"R@{k} (%)")
    ax.set_title(f"R@{k}")
plt.suptitle("Pipeline contribution ladder — each step adds one component")
plt.tight_layout(); plt.show()


### 5B — Qualitative Embedding Traces

**Method**: Three images are selected — most helped, most hurt, and near-median by predictor rank improvement. For each, four embeddings are computed and projected into a shared 2D PCA space (fit on the full projected gallery + all pipeline states): raw image, after avg-condition combiner, after predicted-condition combiner, after oracle combiner. Arrows connect consecutive steps. GT gallery embeddings (red diamonds) mark where the image *should* land for correct retrieval.

**Interpretation**:
- Each arrow = one component's geometric effect on the embedding.
- Arrows pointing *toward* the red diamonds = that component is moving in the right direction.
- The oracle landing point is the best achievable position given the learned combiner.
- For hurt images, look for the arrow that diverges — that is where the pipeline breaks.

**Good**: On the helped image, arrows progressively converge toward GT diamonds. Oracle lands visibly closest to GT. On the hurt image, a clear mis-step is visible (one arrow pointing away or sideways), explaining the failure.

**Bad**: Arrows move randomly with no consistent trend toward GT (combiner has no geometric relationship to the retrieval target). Oracle landing point is far from GT (the combiner cannot reach the target even with the best condition — fundamental architectural limitation).


In [ ]:
# ── 5B: Qualitative embedding traces — where does each component move things? ──
# For a few selected images, trace the embedding through:
#   raw img → combiner(avg cond) → combiner(pred cond) → combiner(oracle cond)
# in a PCA space that also contains the GT text gallery embeddings.
# The GT text embedding is the target; we want to see movement toward it.

if train_types is None or predictor is None or not HAS_CA:
    print("Skipped: needs train_types + predictor + CA cache")
else:
    # Select images: pick a mix — best-improved, worst-hurt, median
    improvement_img = (clip_rank_per_img - pred_rank_per_img).numpy()
    selected_imgs = [
        int(np.argmax(improvement_img)),     # most helped by predictor
        int(np.argmin(improvement_img)),     # most hurt by predictor
        int(np.argsort(np.abs(improvement_img))[n_img // 2]),  # near-median
    ]
    labels_sel = ["most-helped", "most-hurt", "median"]

    # Collect all embeddings we'll project
    cond_avg = type_means.mean(0, keepdim=True)
    all_points, all_labels_str = [], []

    img_states = {}
    for img_idx, lbl in zip(selected_imgs, labels_sel):
        e = all_img_emb[img_idx:img_idx+1]
        # raw
        img_states[img_idx] = {
            "raw":  img_n[img_idx].numpy(),
        }
        with torch.no_grad():
            o_avg = combiner(e.to(dev), None, cond_avg.to(dev)).cpu()
            img_states[img_idx]["avg_cond"] = F.normalize(o_avg, dim=-1)[0].numpy()

            pred_c = pred_conds[img_idx:img_idx+1]
            o_pred = combiner(e.to(dev), None, pred_c.to(dev)).cpu()
            img_states[img_idx]["pred_cond"] = F.normalize(o_pred, dim=-1)[0].numpy()

            best_rep_i = oracle_cond_idx[img2txt[img_idx, 0]].item()
            o_oracle = combiner(e.to(dev), None, representatives[best_rep_i:best_rep_i+1].to(dev)).cpu()
            img_states[img_idx]["oracle_cond"] = F.normalize(o_oracle, dim=-1)[0].numpy()

        # GT text embeddings
        img_states[img_idx]["gt_txts"] = other_n[img2txt[img_idx]].numpy()  # [cpi, 512]

    # Fit PCA on a mix of all involved embeddings + other_n gallery
    fit_data = other_n.numpy()  # gallery (projected texts)
    for s in img_states.values():
        fit_data = np.concatenate([fit_data, [s["raw"], s["avg_cond"], s["pred_cond"], s["oracle_cond"]], s["gt_txts"]])
    pca_trace = PCA(n_components=2)
    pca_trace.fit(fit_data)

    gallery_pc = pca_trace.transform(other_n.numpy())

    fig, axes = plt.subplots(1, len(selected_imgs), figsize=(5 * len(selected_imgs), 5))
    step_styles = {
        "raw":         dict(marker="o", s=120, zorder=5, label="Raw image"),
        "avg_cond":    dict(marker="s", s=100, zorder=5, label="+ avg type-mean cond"),
        "pred_cond":   dict(marker="^", s=100, zorder=5, label="+ pred cond"),
        "oracle_cond": dict(marker="*", s=200, zorder=5, label="+ oracle cond"),
    }
    step_colors = {"raw":"#888888","avg_cond":"#FF9800","pred_cond":"#2196F3","oracle_cond":"#4CAF50"}

    for ax, img_idx, lbl in zip(axes, selected_imgs, labels_sel):
        s = img_states[img_idx]
        # Background: gallery (faint)
        ax.scatter(gallery_pc[:, 0], gallery_pc[:, 1], s=3, alpha=0.1, color="#cccccc")
        # GT texts (bold)
        gt_pc = pca_trace.transform(s["gt_txts"])
        ax.scatter(gt_pc[:, 0], gt_pc[:, 1], s=80, marker="D", color="red", zorder=6, label="GT texts (gallery)")

        # Trace through pipeline steps
        pts = []
        for step, (key, style) in enumerate([("raw",None),("avg_cond",None),("pred_cond",None),("oracle_cond",None)]):
            pt = pca_trace.transform(s[key][np.newaxis])[0]
            pts.append(pt)
            ax.scatter(*pt, color=step_colors[key], **{k:v for k,v in step_styles[key].items() if k!="label"})
        pts = np.array(pts)
        # Draw arrows
        for i in range(len(pts)-1):
            dx, dy = pts[i+1] - pts[i]
            ax.annotate("", xy=pts[i+1], xytext=pts[i],
                        arrowprops=dict(arrowstyle="->", color="#555555", lw=1.2))

        ax.set_title(f"Image #{img_idx} ({lbl})\nΔrank={improvement_img[img_idx]:+.1f}")
        ax.set_xlabel(f"PC1 ({pca_trace.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca_trace.explained_variance_ratio_[1]*100:.1f}%)")

    # Shared legend
    handles = [mpatches.Patch(color=step_colors[k], label=step_styles[k]["label"]) for k in step_styles]
    handles.append(mpatches.Patch(color="red", label="GT texts"))
    axes[-1].legend(handles=handles, fontsize=7, loc="lower right")
    plt.suptitle("Embedding trace through pipeline\n(each arrow = one component applied)")
    plt.tight_layout(); plt.show()
